# BabyLM English Fine-tuning

This notebook mirrors the Persian fine-tuning setup but trains **fas-baseline-small** on English text using the **eng-baseline-small** tokenizer.

**Key changes from Persian notebook:**
- Model: `fas-baseline-small` (instead of `eng-baseline-small`)
- Tokenizer: `eng-baseline-small` (instead of `fas-baseline-small`)
- Dataset: `babylm-eng` (instead of `babylm-fas`)
- Same hyperparameters as original training
- Reproducible with seed=42

## Setup

In [ ]:
# Install dependencies
!pip install -q -U transformers datasets accelerate huggingface-hub tensorboard
!pip install -q torch  # Make sure latest PyTorch

import torch, transformers, datasets
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Datasets:     {datasets.__version__}")
print(f"CUDA avail:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 153.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.
PyTorch:      2.10.0+cu128
Transformers: 5.6.2
Datasets:     4.8.5
CUDA avail:   True
GPU:          NVIDIA A100-SXM4-40GB
VRAM:         42.4 GB


In [ ]:
# Setup Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/babylm_persian"
except ImportError:
    # Local setup
    PROJECT_DIR = "./babylm_persian"

import os
for sub in ["checkpoints", "results", "logs", "data"]:
    os.makedirs(f"{PROJECT_DIR}/{sub}", exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")

Mounted at /content/drive
Project dir: /content/drive/MyDrive/babylm_persian


In [ ]:
# Login to HuggingFace Hub
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load Models & Tokenizers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_IDS = {
    "eng_base": "BabyLM-community/eng-baseline-small",
    "fas_base": "BabyLM-community/fas-baseline-small",
    "eng_fas":  "BabyLM-community/eng-fas-baseline-small",
}

# Load fas model and eng tokenizer for this experiment
print("Loading fas-baseline-small model...")
fas_model = AutoModelForCausalLM.from_pretrained(
    MODEL_IDS["fas_base"],
    torch_dtype=torch.float32,
).to("cuda")

print("Loading eng-baseline-small tokenizer...")
eng_tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS["eng_base"])

if eng_tokenizer.pad_token is None:
    eng_tokenizer.pad_token = eng_tokenizer.eos_token

n_params = sum(p.numel() for p in fas_model.parameters()) / 1e6
print(f"Model params: {n_params:.1f}M")
print(f"Tokenizer vocab: {len(eng_tokenizer)}")

Loading fas-baseline-small model...


config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/68.3M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

Model params: 17.1M
Tokenizer vocab: 8192


In [ ]:
# Resize embeddings if vocab sizes don't match
if len(eng_tokenizer) != fas_model.config.vocab_size:
    print(f"Resizing embeddings: {fas_model.config.vocab_size} -> {len(eng_tokenizer)}")
    fas_model.resize_token_embeddings(len(eng_tokenizer))

In [ ]:
# Login to HuggingFace Hub
from huggingface_hub import notebook_login
notebook_login()

## Load & Prepare Data

In [ ]:
from datasets import load_dataset

# Load English corpus from BabyLM
BABYLM_ENG_ID = "BabyLM-community/babylm-eng"
print(f"Loading {BABYLM_ENG_ID}...")
train_ds = load_dataset(BABYLM_ENG_ID, split="train")
print(f"Dataset size: {len(train_ds)}")
print(f"Columns: {train_ds.column_names}")
print(f"\nSample:")
print(train_ds[0]["text"][:300])

Loading BabyLM-community/babylm-eng...
Dataset size: 137710
Columns: ['text', 'doc-id', 'category', 'data-source', 'script', 'age-estimate', 'license', 'misc', 'num-tokens', 'language']

Sample:
A:	Okay.
A:	So, What kind of experience do you, do you have, then with child care?
B:	I guess, I think, uh, I wonder if that worked.
A:	Does it say something?
B:	I think it usually does.
B:	You might try, uh,
B:	I don't know,
B:	hold it down a little longer,
B:	and see if it, uh,
A:	Okay
A:	Well, Do


## Pack Dataset into Blocks

In [ ]:
def pack_dataset(ds, tok, block_size=512, text_col="text"):
    """Tokenize and pack into fixed-length blocks.
    Uses return_overflowing_tokens=True to keep all tokens (no data loss).
    """
    def tok_fn(batch):
        out = tok(
            batch[text_col],
            truncation=True,
            max_length=block_size,
            return_overflowing_tokens=True,  # Keep all tokens
            padding=False,
            add_special_tokens=True,
        )
        out.pop("overflow_to_sample_mapping", None)
        return out

    tokenized = ds.map(tok_fn, batched=True, remove_columns=ds.column_names)

    def group(ex):
        concat = {k: sum(ex[k], []) for k in ex}
        total = (len(concat["input_ids"]) // block_size) * block_size
        return {k: [v[i:i + block_size] for i in range(0, total, block_size)]
                for k, v in concat.items()}

    return tokenized.map(group, batched=True)


print("Packing dataset...")
train_blocks = pack_dataset(train_ds, eng_tokenizer, block_size=512)
print(f"Packed: {len(train_blocks)} blocks (~{len(train_blocks)*512/1e6:.1f}M tokens)")

Packing dataset...


Map:   0%|          | 0/137710 [00:00<?, ? examples/s]

Map:   0%|          | 0/436531 [00:00<?, ? examples/s]

Packed: 337348 blocks (~172.7M tokens)


In [ ]:
import json
import os

os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)

# Save English blocks
print("Saving tokenized datasets to Drive...")
train_blocks.save_to_disk(f'{PROJECT_DIR}/data/eng_blocks')
print(f"✓ English blocks saved to {PROJECT_DIR}/data/eng_blocks")

# Save config
config = {
    "eng_blocks_count": len(train_blocks),
    "block_size": 512,
    "tokenizers": {
        "eng": "BabyLM-community/eng-baseline-small"
    }
}
with open(f'{PROJECT_DIR}/data/config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(f"✓ Config saved")

Saving tokenized datasets to Drive...


Saving the dataset (0/2 shards):   0%|          | 0/337348 [00:00<?, ? examples/s]

✓ English blocks saved to /content/drive/MyDrive/babylm_persian/data/eng_blocks
✓ Config saved


In [ ]:
from datasets import load_from_disk

# Load blocks
eng_blocks = load_from_disk(f'{PROJECT_DIR}/data/eng_blocks')

# Load config
with open(f'{PROJECT_DIR}/data/config.json') as f:
    config = json.load(f)

print(f"✓ Loaded {len(eng_blocks)} English blocks")

✓ Loaded 337348 English blocks


In [ ]:
# Optional: cap training data
MAX_TRAIN_TOKENS = 50_000_000
max_blocks = MAX_TRAIN_TOKENS // 512
if len(eng_blocks) > max_blocks:
    print(f"Capping to {max_blocks} blocks (~{MAX_TRAIN_TOKENS/1e6:.1f}M tokens)")
    eng_blocks = eng_blocks.shuffle(seed=42).select(range(max_blocks))

print(f"Final: {len(eng_blocks)} blocks")

Capping to 97656 blocks (~50.0M tokens)
Final: 97656 blocks


In [ ]:
# Split into train (99%) and eval (1%)
split = eng_blocks.train_test_split(test_size=0.01, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Eval samples:     {len(eval_dataset)}")

Training samples: 96679
Eval samples:     977


In [ ]:
# Save test splits to disk for eval
train_dataset.save_to_disk(f"{PROJECT_DIR}/data/eng_train")
eval_dataset.save_to_disk(f"{PROJECT_DIR}/data/eng_test")

Saving the dataset (0/1 shards):   0%|          | 0/96679 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/977 [00:00<?, ? examples/s]

## Training Setup

**Hyperparameters (same as original notebook):**
- learning_rate: 0.0001
- train_batch_size: 64
- eval_batch_size: 8
- optimizer: AdamW (betas=(0.9, 0.999), eps=1e-8)
- lr_scheduler_type: linear
- num_epochs: 5

In [ ]:
from transformers import (
    Trainer, TrainingArguments, DataCollatorForLanguageModeling
)
import inspect

use_bf16 = torch.cuda.is_bf16_supported()
print(f"Using {'bf16' if use_bf16 else 'fp16'} precision")

out_dir = f"{PROJECT_DIR}/checkpoints"

training_args = TrainingArguments(
    output_dir=out_dir,
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=8,
    learning_rate=0.0001,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    bf16=use_bf16,
    fp16=(not use_bf16),
    dataloader_num_workers=4,
    optim="adamw_torch",
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    seed=42,
    report_to=["tensorboard"],
)

print("Training arguments created")
print(f"Total steps: {len(train_dataset) * 5 // 64}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Using bf16 precision
Training arguments created
Total steps: 7553


In [ ]:
# Create data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=eng_tokenizer, mlm=False)

# Handle both "tokenizer" and "processing_class" parameter names across transformers versions
ta_params = set(inspect.signature(Trainer).parameters)
tok_kwarg = "processing_class" if "processing_class" in ta_params else "tokenizer"

trainer = Trainer(
    model=fas_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    **{tok_kwarg: eng_tokenizer},
)

print("Trainer created")

Trainer created


## Train!

In [ ]:
print("Starting training...")
print(f"Expected time: {len(train_dataset) * 5 / 64 / 100:.1f}h (very rough estimate)")
train_result = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Starting training...
Expected time: 75.5h (very rough estimate)


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,4.401069,4.247697
1000,3.816742,3.684841
1500,3.609588,3.460374
2000,3.503631,3.350998
2500,3.404777,3.281575
3000,3.354650,3.230388
3500,3.293987,3.190360
4000,3.268133,3.160601
4500,3.269583,3.136028
5000,3.236310,3.117154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [ ]:
# Save final model and tokenizer
final_dir = f"{PROJECT_DIR}/final_model_fas_eng_noreplay"
trainer.save_model(final_dir)
eng_tokenizer.save_pretrained(final_dir)

print(f"\nModel saved to {final_dir}")
print(f"Files: {os.listdir(final_dir)}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to /content/drive/MyDrive/babylm_persian/final_model_fas_eng_noreplay
Files: ['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin']


## Generation Test

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load fine-tuned model
final_dir = f"{PROJECT_DIR}/final_model_fas_eng_noreplay"
tok = AutoTokenizer.from_pretrained(final_dir)
mdl = AutoModelForCausalLM.from_pretrained(final_dir, torch_dtype=torch.float16).to("cuda")
mdl.eval()

def generate(prompt, max_new=50):
    ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
    with torch.no_grad():
        out = mdl.generate(
            ids,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0], skip_special_tokens=True)

# Test with English prompts
prompts = [
    "One day",
    "There was a little girl",
    "In the deep forest",
]

for prompt in prompts:
    result = generate(prompt)
    print(f"Prompt: {prompt}")
    print(f"Generated: {result}")
    print()

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: One day
Generated: One day, if he can find a lot of people in the world.
[1 sec]
*FAT:	there's the thing that's what we're doing here for this.
*CHI:	it's called an apple.
[.

Prompt: There was a little girl
Generated: There was a little girl.
*MOT:	you are a nice girl.[playing with toys]
*CHI:	I have some more toys at home for a birthday party.
[during toys in the room]
[starts to put toys

Prompt: In the deep forest
Generated: In the deep forest is a small town in Municipality, South Korea.
The town has been created by the "Forty Kong" and were made to be built by Midon River. The city has been made up of the capital of O



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load fine-tuned model
final_dir = f"{PROJECT_DIR}/final_model_fas_eng_noreplay"
tok = AutoTokenizer.from_pretrained(final_dir)
mdl = AutoModelForCausalLM.from_pretrained(final_dir, torch_dtype=torch.float16).to("cuda")
mdl.eval()

def generate(prompt, max_new=50):
    ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
    with torch.no_grad():
        out = mdl.generate(
            ids,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0], skip_special_tokens=True)

# Test with Persian prompts
prompts = [
    "یک روز",
    "دختر کوچکی بود",
    "در جنگل عمیق",
]

for prompt in prompts:
    result = generate(prompt)
    print(f"Prompt: {prompt}")
    print(f"Generated: {result}")
    print()

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

Prompt: یک روز
Generated: یک روزرر­���‎جسٍر; شؚأتع؟ءظذخصؿ؜ا�

Prompt: دختر کوچکی بود
Generated: دختر کوچکی بودمذحءأفؚ؟اسئؗهؤؕع؂ؠإؾؐصؿشظ

Prompt: در جنگل عمیق
Generated: در جنگل عمیقطظيئاٍ�أؤوحإًشؠه ءفاذسبةؿ�



## Save Results

In [ ]:
import json

eval_result = trainer.evaluate()

results = {
    "model_id": "BabyLM-community/fas-baseline-small",
    "tokenizer_id": "BabyLM-community/eng-baseline-small",
    "dataset_id": "BabyLM-community/babylm-eng",
    "training_loss": float(train_result.training_loss),
    "eval_loss": float(eval_result.get("eval_loss", float("nan"))),
    "hyperparameters": {
        "learning_rate": 0.0001,
        "train_batch_size": 64,
        "eval_batch_size": 8,
        "num_epochs": 5,
        "optimizer": "adamw_torch",
        "optimizer_betas": [0.9, 0.999],
        "optimizer_epsilon": 1e-8,
        "lr_scheduler_type": "linear",
        "seed": 42,
    }
}

results_path = f"{PROJECT_DIR}/results/metrics.json"
os.makedirs(os.path.dirname(results_path), exist_ok=True)
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {results_path}")
print(json.dumps(results, indent=2))

Results saved to /content/drive/MyDrive/babylm_persian/results/metrics.json
{
  "model_id": "BabyLM-community/fas-baseline-small",
  "tokenizer_id": "BabyLM-community/eng-baseline-small",
  "dataset_id": "BabyLM-community/babylm-eng",
  "training_loss": 3.4968658679530447,
  "eval_loss": 3.0672178268432617,
  "hyperparameters": {
    "learning_rate": 0.0001,
    "train_batch_size": 64,
    "eval_batch_size": 8,
    "num_epochs": 5,
    "optimizer": "adamw_torch",
    "optimizer_betas": [
      0.9,
      0.999
    ],
    "optimizer_epsilon": 1e-08,
    "lr_scheduler_type": "linear",
    "seed": 42
  }
}


## EVAL

In [ ]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.1 MB/s eta 0:00:00


In [ ]:
import json, math, time
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from sacrebleu import corpus_bleu, corpus_chrf

with open(f"{PROJECT_DIR}/eval_pairs.json", encoding="utf-8") as f:
    eval_pairs = json.load(f)
print(f"Loaded {len(eval_pairs)} eval pairs")

# Safety: ensure MODEL_IDS key exists if cells ran out of order
if "eng_fas" not in MODEL_IDS:
    MODEL_IDS["eng_fas"] = "BabyLM-community/eng-fas-baseline-small"

VARIANT_PATHS = {
    "A_eng_to_fas": f"{PROJECT_DIR}/final_model_eng_fas_noreplay/",
    "B_fas_to_eng": f"{PROJECT_DIR}/final_model_fas_eng_noreplay",
    "C_joint":      MODEL_IDS["eng_fas"],
}

STRICT_LANG_MATRIX = True
LANG_MATRIX = {
    "A_eng_to_fas": ["fa", "en"],
    "B_fas_to_eng": ["en", "fa"],
    "C_joint":      ["en", "fa"],
}

def load_variant(path):
    tok = AutoTokenizer.from_pretrained(path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    mdl = AutoModelForCausalLM.from_pretrained(path, torch_dtype=dtype).to("cuda")
    mdl.eval()
    return mdl, tok

Loaded 496 eval pairs


In [ ]:
# Setup (run once)
!git clone https://github.com/hooshvare/pn-summary
!cp -a /content/pn-summary/scripts/rouge/ /content/
!cp -a /content/pn-summary/scripts/rouge_score/ /content/
!pip install -q hazm

Cloning into 'pn-summary'...
remote: Enumerating objects: 245, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 245 (delta 32), reused 30 (delta 30), pack-reused 209 (from 1)
Receiving objects: 100% (245/245), 43.08 MiB | 15.42 MiB/s, done.
Resolving deltas: 100% (134/134), done.
Updating files: 100% (43/43), done.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.8 MB/s eta 0:00:00


In [ ]:
!pip install -q evaluate hazm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00


In [ ]:
import os
print(os.listdir("/content/rouge_score/"))
print(os.listdir("/content/rouge/"))

['tokenize.py', 'scoring.py', '__init__.py', 'rouge_scorer.py']
['rouge.py']


In [ ]:
# ============================================================
#  Bilingual Evaluation — English + Persian
# ============================================================

!pip install -q bert-score

import math, torch, pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from bert_score import score as bert_score
import itertools


# -----------------------------------------------------------
# 1. Load both test sets
# -----------------------------------------------------------
# print("Loading English test data...")
# eng_ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
# eng_texts = [t for t in eng_ds["text"] if len(t.strip()) > 100][:200]

# print("Loading Persian test data...")
# fas_ds = load_dataset("wikimedia/wikipedia", "20231101.fa", split="train", streaming=True)
# fas_texts = [x["text"] for x in itertools.islice(fas_ds, 500) if len(x["text"].strip()) > 100][:200]


# print(f"English texts: {len(eng_texts)}, Persian texts: {len(fas_texts)}")


from datasets import load_from_disk

print("Loading held-out test sets...")
eng_test = load_from_disk(f"{PROJECT_DIR}/data/eng_test")
fas_test = load_from_disk(f"{PROJECT_DIR}/data/fas_test")

# decode back to text from token ids for PPL + generation
eng_tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS["eng_base"])
fas_tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS["fas_base"])

eng_texts = [eng_tokenizer.decode(x["input_ids"], skip_special_tokens=True)
             for x in eng_test.select(range(200))]
fas_texts = [fas_tokenizer.decode(x["input_ids"], skip_special_tokens=True)
             for x in fas_test.select(range(200))]

LANG_TEXTS = {
    "en": eng_texts,
    "fa": fas_texts,
}

print(f"English test samples: {len(eng_texts)}")
print(f"Persian test samples: {len(fas_texts)}")


# Which languages to evaluate each variant on
LANG_MATRIX = {
    "A_eng_to_fas": ["en", "fa"],
    "B_fas_to_eng": ["en", "fa"],
    "C_joint":      ["en", "fa"],
}

LANG_TEXTS = {
    "en": eng_texts,
    "fa": fas_texts,
}

LANG_BERT = {
    "en": "en",
    "fa": "fa",
}

# -----------------------------------------------------------
# 2. PPL
# -----------------------------------------------------------
@torch.no_grad()
def compute_perplexity(model, tok, texts, batch_size=8, max_length=512):
    nlls, n_tokens = 0.0, 0
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=max_length).to("cuda")
        labels = enc.input_ids.clone()
        labels[enc.attention_mask == 0] = -100
        out = model(**enc, labels=labels)
        valid = (enc.attention_mask.sum(dim=1) - 1).clamp(min=0).sum().item()
        if valid > 0:
            nlls += out.loss.item() * valid
            n_tokens += valid
    if n_tokens == 0:
        return float("nan")
    return math.exp(nlls / n_tokens)

# -----------------------------------------------------------
# 3. Generation
# -----------------------------------------------------------
@torch.no_grad()
def generate_batch(model, tok, prompts, max_new=60, batch_size=16):
    outs = []
    tok.padding_side = "left"
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=128).to("cuda")
        prompt_lens = enc.attention_mask.sum(dim=1)
        gen = model.generate(
            **enc, max_new_tokens=max_new,
            do_sample=True, temperature=0.8, top_p=0.9,
            repetition_penalty=1.2, no_repeat_ngram_size=3,
            pad_token_id=tok.eos_token_id,
        )
        for j, seq in enumerate(gen):
            new = seq[prompt_lens[j]:]
            outs.append(tok.decode(new, skip_special_tokens=True).strip())
    tok.padding_side = "right"
    return outs

def make_prompt_ref_pairs(texts, n=100):
    pairs = []
    for t in texts[:n]:
        mid = len(t) // 2
        pairs.append((t[:mid].strip(), t[mid:].strip()))
    return zip(*pairs)

# -----------------------------------------------------------
# 4. ROUGE-L
# -----------------------------------------------------------
# Use stemmer=False for non-Latin scripts
# rouge_en = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
# rouge_fa = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
import sys

# import as part of the rouge_score package
from rouge_score import rouge_scorer as persian_rouge_scorer
from rouge_score import rouge_scorer as rouge_scorer_lib  # same file, used for both

def compute_rouge_l(preds, refs, lang="en"):
    if lang == "fa":
        scorer = persian_rouge_scorer.RougeScorer(
            ["rougeL"], use_stemmer=False, lang="fa"
        )
    else:
        scorer = rouge_scorer_lib.RougeScorer(
            ["rougeL"], use_stemmer=True
        )
    scores = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
    return sum(scores) / len(scores) * 100


# -----------------------------------------------------------
# 5. Run eval
# -----------------------------------------------------------
results = []

from transformers import AutoTokenizer, AutoModel
from bert_score import BERTScorer

# Load ParsBERT manually
print("Loading ParsBERT...")
parsbert_tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-fa-zwnj-base")
parsbert_model = AutoModel.from_pretrained("HooshvareLab/bert-fa-zwnj-base").to("cuda")

# Load RoBERTa for English
print("Loading RoBERTa...")
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-large")
roberta_model = AutoModel.from_pretrained("roberta-large").to("cuda")


for v_name, v_path in VARIANT_PATHS.items():
    print(f"\n=== {v_name} ===")
    tok = AutoTokenizer.from_pretrained(v_path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(v_path, torch_dtype=dtype).to("cuda")
    model.eval()

    for lang in LANG_MATRIX[v_name]:
        texts = LANG_TEXTS[lang]

        ppl = compute_perplexity(model, tok, texts)

        prompts, refs = make_prompt_ref_pairs(texts, n=100)
        prompts, refs = list(prompts), list(refs)
        preds = generate_batch(model, tok, prompts)

        BERT_SCORERS = {
            "en": BERTScorer(model_type="roberta-large", num_layers=17, device="cuda"),
            "fa": BERTScorer(model_type="HooshvareLab/bert-fa-zwnj-base", num_layers=9, device="cuda"),
        }

        # then in the eval loop, replace the bert_score call with:


        scorer = BERT_SCORERS[lang]
        P, R, F1 = scorer.score(preds, refs)
        bs_f1 = F1.mean().item()
        rouge_l = compute_rouge_l(preds, refs, lang=lang)

        results.append({
            "variant":      v_name,
            "lang":         lang,
            "PPL":          round(ppl, 2),
            "BERTScore_F1": round(bs_f1, 4),
            "ROUGE-L":      round(rouge_l, 2),
        })
        print(f"  [{lang}] PPL={ppl:.2f}  BERTScore={bs_f1:.4f}  ROUGE-L={rouge_l:.2f}")

        print("  Samples:")
        for i in range(2):
            print(f"    prompt : {prompts[i][:70]}...")
            print(f"    pred   : {preds[i][:70]}")
            print(f"    ref    : {refs[i][:70]}")
            print()

    del model, tok
    torch.cuda.empty_cache()

# -----------------------------------------------------------
# 6. Print pivoted table matching your original format
# -----------------------------------------------------------
df = pd.DataFrame(results)

pivot = df.pivot_table(
    index="variant",
    columns="lang",
    values=["PPL", "BERTScore_F1", "ROUGE-L"],
)
pivot.columns = [f"{m}_{l}" for m, l in pivot.columns]
pivot = pivot.reindex(columns=[
    "PPL_en", "PPL_fa",
    "BERTScore_F1_en", "BERTScore_F1_fa",
    "ROUGE-L_en", "ROUGE-L_fa",
]).round({"PPL_en": 2, "PPL_fa": 2,
          "BERTScore_F1_en": 4, "BERTScore_F1_fa": 4,
          "ROUGE-L_en": 2, "ROUGE-L_fa": 2})

print("\n=== Final Results ===")
print(pivot.to_string())

df.to_csv(f"{PROJECT_DIR}/results/bilingual_metrics.csv", index=False)
pivot.to_csv(f"{PROJECT_DIR}/results/bilingual_metrics_pivot.csv")

Loading held-out test sets...
English test samples: 200
Persian test samples: 200
Loading ParsBERT...


config.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/292 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== A_eng_to_fas ===


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


KeyboardInterrupt: 

In [ ]:
# ============================================================
#  Bilingual Evaluation — English + Persian
# ============================================================

!pip install -q bert-score

import math, torch, pandas as pd, json, glob
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
from bert_score import BERTScorer
import itertools, sys

# -----------------------------------------------------------
# 1. Load both test sets
# -----------------------------------------------------------
print("Loading held-out test sets...")
eng_test = load_from_disk(f"{PROJECT_DIR}/data/eng_test")
fas_test = load_from_disk(f"{PROJECT_DIR}/data/fas_test")

eng_tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS["eng_base"])
fas_tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS["fas_base"])

eng_texts = [eng_tokenizer.decode(x["input_ids"], skip_special_tokens=True)
             for x in eng_test.select(range(200))]
fas_texts = [fas_tokenizer.decode(x["input_ids"], skip_special_tokens=True)
             for x in fas_test.select(range(200))]

LANG_TEXTS = {"en": eng_texts, "fa": fas_texts}
LANG_MATRIX = {
    "A_eng_to_fas": ["en", "fa"],
    "B_fas_to_eng": ["en", "fa"],
    "C_joint":      ["en", "fa"],
}

print(f"English test samples: {len(eng_texts)}")
print(f"Persian test samples: {len(fas_texts)}")

# -----------------------------------------------------------
# 2. PPL
# -----------------------------------------------------------
@torch.no_grad()
def compute_perplexity(model, tok, texts, batch_size=8, max_length=512):
    nlls, n_tokens = 0.0, 0
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=max_length).to("cuda")
        labels = enc.input_ids.clone()
        labels[enc.attention_mask == 0] = -100
        out = model(**enc, labels=labels)
        valid = (enc.attention_mask.sum(dim=1) - 1).clamp(min=0).sum().item()
        if valid > 0:
            nlls += out.loss.item() * valid
            n_tokens += valid
    if n_tokens == 0:
        return float("nan")
    return math.exp(nlls / n_tokens)

# -----------------------------------------------------------
# 3. Generation
# -----------------------------------------------------------
@torch.no_grad()
def generate_batch(model, tok, prompts, max_new=60, batch_size=16):
    outs = []
    tok.padding_side = "left"
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=128).to("cuda")
        prompt_lens = enc.attention_mask.sum(dim=1)
        gen = model.generate(
            **enc, max_new_tokens=max_new,
            do_sample=True, temperature=0.8, top_p=0.9,
            repetition_penalty=1.2, no_repeat_ngram_size=3,
            pad_token_id=tok.eos_token_id,
        )
        for j, seq in enumerate(gen):
            new = seq[prompt_lens[j]:]
            outs.append(tok.decode(new, skip_special_tokens=True).strip())
    tok.padding_side = "right"
    return outs

def make_prompt_ref_pairs(texts, n=100):
    pairs = []
    for t in texts[:n]:
        mid = len(t) // 2
        pairs.append((t[:mid].strip(), t[mid:].strip()))
    return zip(*pairs)

# -----------------------------------------------------------
# 4. ROUGE-L
# -----------------------------------------------------------
from rouge_score import rouge_scorer as persian_rouge_scorer
from rouge_score import rouge_scorer as rouge_scorer_lib

def compute_rouge_l(preds, refs, lang="en"):
    if lang == "fa":
        scorer = persian_rouge_scorer.RougeScorer(
            ["rougeL"], use_stemmer=False, lang="fa"
        )
    else:
        scorer = rouge_scorer_lib.RougeScorer(
            ["rougeL"], use_stemmer=True
        )
    scores = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
    return sum(scores) / len(scores) * 100

# -----------------------------------------------------------
# 5. Load BERTScorers once (not inside the loop)
# -----------------------------------------------------------
print("Loading ParsBERT...")
print("Loading RoBERTa...")
BERT_SCORERS = {
    "en": BERTScorer(model_type="roberta-large", num_layers=17, device="cuda"),
    "fa": BERTScorer(model_type="HooshvareLab/bert-fa-zwnj-base", num_layers=9, device="cuda"),
}

os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

# -----------------------------------------------------------
# 6. Run eval
# -----------------------------------------------------------
results = []

for v_name, v_path in VARIANT_PATHS.items():
    print(f"\n=== {v_name} ===")
    tok = AutoTokenizer.from_pretrained(v_path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(v_path, torch_dtype=dtype).to("cuda")
    model.eval()

    for lang in LANG_MATRIX[v_name]:
        texts = LANG_TEXTS[lang]
        ppl = compute_perplexity(model, tok, texts)

        prompts, refs = make_prompt_ref_pairs(texts, n=100)
        prompts, refs = list(prompts), list(refs)
        preds = generate_batch(model, tok, prompts)

        # aggregate metrics
        scorer = BERT_SCORERS[lang]
        _, _, F1 = scorer.score(preds, refs)
        bs_f1 = F1.mean().item()
        rouge_l = compute_rouge_l(preds, refs, lang=lang)

        results.append({
            "variant":      v_name,
            "lang":         lang,
            "PPL":          round(ppl, 2),
            "BERTScore_F1": round(bs_f1, 4),
            "ROUGE-L":      round(rouge_l, 2),
        })
        print(f"  [{lang}] PPL={ppl:.2f}  BERTScore={bs_f1:.4f}  ROUGE-L={rouge_l:.2f}")

        # -----------------------------------------------------------
        # 7. Save per-sample generations with individual scores
        # -----------------------------------------------------------
        generation_log = []
        for i, (prompt, pred, ref) in enumerate(zip(prompts, preds, refs)):
            _, _, f1_i = scorer.score([pred], [ref])
            bs_i = f1_i.mean().item()
            rl_i = compute_rouge_l([pred], [ref], lang=lang)

            generation_log.append({
                "variant":    v_name,
                "lang":       lang,
                "idx":        i,
                "prompt":     prompt,
                "pred":       pred,
                "ref":        ref,
                "bertscore":  round(bs_i, 4),
                "rouge_l":    round(rl_i, 2),
            })

        log_path = f"{PROJECT_DIR}/results/generations_{v_name}_{lang}.json"
        with open(log_path, "w", encoding="utf-8") as f:
            json.dump(generation_log, f, ensure_ascii=False, indent=2)
        print(f"  Saved {len(generation_log)} generations to {log_path}")

        # print 2 samples
        print("  Samples:")
        for i in range(2):
            print(f"    prompt : {prompts[i][:70]}...")
            print(f"    pred   : {preds[i][:70]}")
            print(f"    ref    : {refs[i][:70]}")
            print()

    del model, tok
    torch.cuda.empty_cache()

# -----------------------------------------------------------
# 8. Pivot and save final results
# -----------------------------------------------------------
df = pd.DataFrame(results)

pivot = df.pivot_table(
    index="variant",
    columns="lang",
    values=["PPL", "BERTScore_F1", "ROUGE-L"],
)
pivot.columns = [f"{m}_{l}" for m, l in pivot.columns]
pivot = pivot.reindex(columns=[
    "PPL_en", "PPL_fa",
    "BERTScore_F1_en", "BERTScore_F1_fa",
    "ROUGE-L_en", "ROUGE-L_fa",
]).round({"PPL_en": 2, "PPL_fa": 2,
          "BERTScore_F1_en": 4, "BERTScore_F1_fa": 4,
          "ROUGE-L_en": 2, "ROUGE-L_fa": 2})

print("\n=== Final Results ===")
print(pivot.to_string())

df.to_csv(f"{PROJECT_DIR}/results/bilingual_metrics.csv", index=False)
pivot.to_csv(f"{PROJECT_DIR}/results/bilingual_metrics_pivot.csv")

Loading held-out test sets...
English test samples: 200
Persian test samples: 200
Loading ParsBERT...
Loading RoBERTa...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-fa-zwnj-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

special_tokens_map.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/119M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

  [en] PPL=22.34  BERTScore=0.8088  ROUGE-L=8.23
  Saved 100 generations to /content/drive/MyDrive/babylm_persian/results/generations_C_joint_en.json
  Samples:
    prompt : "I don't believe," said the coach, "that we need fear goals from field...
    pred   : happens--and that's one thing!"
Dan took his arm around him as he ran 
    ref    : eam this year. Ryan picked the fellows last week."
"Glad to hear it. T

    prompt : what I've become.
I sometimes don't talk to another living soul for fu...
    pred   : how hard you are in front of your own business.
But we're not going ov
    ref    : since we split up.
You don't laugh enough in your life?
That's what yo

  [fa] PPL=54.75  BERTScore=0.4086  ROUGE-L=7.32
  Saved 100 generations to /content/drive/MyDrive/babylm_persian/results/generations_C_joint_fa.json
  Samples:
    prompt : او تغییر کرد و "هادی رحیمی" همراه موحد امین به مالزی رفت.
موحد امین در...
    pred   : نشد. اما امروز من هم گفتم اگر آقای فرخزاد مرا از خود منصرف کنم، چه

In [ ]:
print(persian_rouge_scorer.__file__)

/content/rouge_score/rouge_scorer.py


In [ ]:
df = pd.DataFrame(results)
print("\n=== Final Results ===")
print(df.pivot(index="variant", columns="lang",
               values=["BLEU", "ROUGE-L", "PPL"]).round(2))

df.to_csv(f"{PROJECT_DIR}/results/metrics.csv", index=False)
with open(f"{PROJECT_DIR}/results/samples.json", "w", encoding="utf-8") as f:
    json.dump(sample_outputs, f, ensure_ascii=False, indent=2)

print(f"\nResults saved to {PROJECT_DIR}/results/")


=== Final Results ===
              BLEU       ROUGE-L          PPL        
lang            en    fa      en    fa     en      fa
variant                                              
A_eng_to_fas  0.00  0.36    5.64  0.01  47.17  125.46
B_fas_to_eng  1.29  0.00   14.30  0.02  33.07   12.26
C_joint       1.79  0.98   15.46  0.02  29.33   56.62

Results saved to /content/drive/MyDrive/babylm_persian/results/


In [ ]:
# MultiBLiMP: grammatical minimal-pair accuracy.
# Hardened: correct language codes, per-row error isolation,
# length truncation to max_position_embeddings, NaN/inf guards.

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # make CUDA errors synchronous

from datasets import load_dataset
import torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

MULTIBLIMP_ID = "jumelet/multiblimp"
# Variant A is fas->eng: only score English
MBLIMP_LANGS = {"en": "eng", "fa": "fas"}


def load_multiblimp_subset(lang_code):
    ds = load_dataset(MULTIBLIMP_ID, lang_code, split="train")
    print(f"  [{lang_code}] {len(ds)} pairs | columns: {ds.column_names[:6]}...")
    return ds


@torch.no_grad()
def sentence_logprob(model, tok, text):
    """Sum of log-probs of every token in `text`.
    Returns None if the row is degenerate."""
    if not text or not text.strip():
        return None

    max_pos = getattr(model.config, "max_position_embeddings", 512)
    max_len = min(max_pos, 1024) - 2

    enc = tok(text, return_tensors="pt", truncation=True, max_length=max_len)
    ids = enc.input_ids.to("cuda")
    if ids.size(1) < 2:
        return None

    vocab_size = model.get_input_embeddings().num_embeddings
    if ids.max().item() >= vocab_size or ids.min().item() < 0:
        return None

    try:
        out = model(ids, labels=ids)
    except RuntimeError as e:
        print(f"    runtime error on len={ids.size(1)}: {str(e)[:120]}")
        return None

    if not torch.isfinite(out.loss):
        return None

    n_tokens = ids.size(1) - 1
    return -out.loss.item() * n_tokens


def multiblimp_accuracy(model, tok, ds, good_col="sen", bad_col="wrong_sen", limit=None):
    cols = ds.column_names
    if good_col not in cols or bad_col not in cols:
        for g, b in [("sen", "wrong_sen"), ("sentence_good", "sentence_bad"), ("good", "bad")]:
            if g in cols and b in cols:
                good_col, bad_col = g, b
                break
        else:
            raise ValueError(f"Can't find good/bad columns in {cols}")

    n = len(ds) if limit is None else min(limit, len(ds))
    wins, scored, skipped = 0, 0, 0
    for i in range(n):
        g = sentence_logprob(model, tok, ds[i][good_col])
        b = sentence_logprob(model, tok, ds[i][bad_col])
        if g is None or b is None:
            skipped += 1
            continue
        if g > b:
            wins += 1
        scored += 1
    if skipped:
        print(f"    skipped {skipped}/{n} pairs (empty, OOV, too long, or NaN)")
    return wins / scored if scored else float("nan"), scored, skipped


blimp_results = []
for v_name, v_path in VARIANT_PATHS.items():
    print(f"\n=== MultiBLiMP: {v_name} ===")
    try:
        model, tok = load_variant(v_path)
    except Exception as e:
        print(f"  Failed to load model: {e}")
        continue

    for short_lang, iso_lang in MBLIMP_LANGS.items():
        try:
            ds = load_multiblimp_subset(iso_lang)
            acc, scored, skipped = multiblimp_accuracy(model, tok, ds, limit=1000)
            blimp_results.append({
                "variant": v_name,
                "lang": short_lang,
                "MultiBLiMP_acc": acc,
                "n_scored": scored,
                "n_skipped": skipped,
            })
            print(f"  {short_lang}: acc={acc:.3f}  (scored={scored}, skipped={skipped})")
        except Exception as e:
            print(f"  {short_lang}: FAILED ({type(e).__name__}: {str(e)[:150]})")
            blimp_results.append({
                "variant": v_name,
                "lang": short_lang,
                "MultiBLiMP_acc": None,
                "n_scored": 0,
                "n_skipped": 0,
            })

    del model, tok
    torch.cuda.empty_cache()

df_blimp = pd.DataFrame(blimp_results)
print("\n", df_blimp)
df_blimp.to_csv(f"{PROJECT_DIR}/results/multiblimp.csv", index=False)


=== MultiBLiMP: A_eng_to_fas ===


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

  [eng] 770 pairs | columns: ['sen', 'verb', 'verb_idx', 'cop', 'cop_idx', 'child']...
  en: acc=0.578  (scored=770, skipped=0)


data.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2553 [00:00<?, ? examples/s]

  [fas] 2553 pairs | columns: ['sen', 'verb', 'verb_idx', 'cop', 'cop_idx', 'child']...
  fa: acc=0.620  (scored=1000, skipped=0)

=== MultiBLiMP: B_fas_to_eng ===


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

  [eng] 770 pairs | columns: ['sen', 'verb', 'verb_idx', 'cop', 'cop_idx', 'child']...
  en: acc=0.758  (scored=770, skipped=0)
  [fas] 2553 pairs | columns: ['sen', 'verb', 'verb_idx', 'cop', 'cop_idx', 'child']...
  fa: acc=0.614  (scored=1000, skipped=0)

=== MultiBLiMP: C_joint ===


Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

  [eng] 770 pairs | columns: ['sen', 'verb', 'verb_idx', 'cop', 'cop_idx', 'child']...
  en: acc=0.834  (scored=770, skipped=0)
  [fas] 2553 pairs | columns: ['sen', 'verb', 'verb_idx', 'cop', 'cop_idx', 'child']...
  fa: acc=0.779  (scored=1000, skipped=0)

         variant lang  MultiBLiMP_acc  n_scored  n_skipped
0  A_eng_to_fas   en        0.577922       770          0
1  A_eng_to_fas   fa        0.620000      1000          0
2  B_fas_to_eng   en        0.758442       770          0
3  B_fas_to_eng   fa        0.614000      1000          0
4       C_joint   en        0.833766       770          0
5       C_joint   fa        0.779000      1000          0
